#Pertemuan 2: Data Preprocessing dan EDA

In [1]:
import matplotlib
matplotlib.use("Agg") # Mode non-interaktif untuk penyimpanan file

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ============================================================
# Memuat dataset Titanic
# ============================================================
df = sns.load_dataset("titanic")
print("=== Ukuran Data Awal ===")
print(f"Shape: {df.shape}")
print("\nTipe kolom:")
print(df.dtypes)

=== Ukuran Data Awal ===
Shape: (891, 15)

Tipe kolom:
survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object


In [2]:
# 1. Missing Value
print("\n=== Missing Value (sebelum) ===")
missing_before = df.isnull().sum()
print(missing_before[missing_before > 0])

# Strategi pengisian:
# - Age: Median (karena distribusi mungkin miring)
# - Embarked/Embark Town: Mode (kategori paling sering muncul)
# - Deck: Kategori baru "Unknown" (karena terlalu banyak yang hilang)
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

# Penanganan khusus untuk kolom kategori 'deck'
if 'deck' in df.columns:
    df["deck"] = df["deck"].cat.add_categories("Unknown").fillna("Unknown")

print("\n=== Missing Value (sesudah) ===")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else "Tidak ada missing value tersisa.")


=== Missing Value (sebelum) ===
age            177
embarked         2
deck           688
embark_town      2
dtype: int64

=== Missing Value (sesudah) ===
Tidak ada missing value tersisa.


In [3]:
# 2. Duplikat
print("\n=== Duplikat ===")
duplicates_count = df.duplicated().sum()
print(f"Jumlah duplikat ditemukan: {duplicates_count}")
df = df.drop_duplicates()
print(f"Shape setelah hapus duplikat: {df.shape}")


=== Duplikat ===
Jumlah duplikat ditemukan: 110
Shape setelah hapus duplikat: (781, 15)


In [4]:
# 3. Outlier (IQR) pada kolom fare dan age
print("\n=== Outlier (IQR) ===")
for col in ["fare", "age"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    outliers_before = ((df[col] < low) | (df[col] > high)).sum()
    # Clipping/Capping: Membatasi nilai ke batas low/high
    df[col] = df[col].clip(low, high)
    print(f"  {col.capitalize()}: {outliers_before} outlier diproses (batas {low:.1f} – {high:.1f})")


=== Outlier (IQR) ===
  Fare: 102 outlier diproses (batas -30.9 – 73.0)
  Age: 39 outlier diproses (batas 1.0 – 57.0)


In [5]:
# 4. Scaling (Standardization)
# Mengubah distribusi menjadi mean=0 dan std=1
scaler = StandardScaler()
df[["age", "fare"]] = scaler.fit_transform(df[["age", "fare"]])

print("\n=== Statistik Setelah Scaling ===")
print(df[["age", "fare"]].describe().round(3))


=== Statistik Setelah Scaling ===
           age     fare
count  781.000  781.000
mean     0.000   -0.000
std      1.001    1.001
min     -2.170   -1.167
25%     -0.562   -0.812
50%     -0.103   -0.466
75%      0.510    0.333
max      2.117    2.050


In [6]:
# 5. EDA — 5 Visualisasi + 5 Insight
# ============================================================

# Menyiapkan canvas visualisasi
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
sns.set_style("whitegrid")

# Visualisasi 1: Distribusi Target (Survived)
df["survived"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#e74c3c", "#2ecc71"])
axes[0, 0].set_title("1. Distribusi Survived")
axes[0, 0].set_xticklabels(["Tidak Selamat", "Selamat"], rotation=0)
axes[0, 0].set_ylabel("Jumlah Penumpang")

# Visualisasi 2: Distribusi Umur (Setelah Scaling)
sns.histplot(df["age"], kde=True, ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("2. Distribusi Age (Standardized)")

# Visualisasi 3: Distribusi Fare (Setelah Scaling)
sns.histplot(df["fare"], kde=True, ax=axes[0, 2], color="coral")
axes[0, 2].set_title("3. Distribusi Fare (Standardized)")

# Visualisasi 4: Survival Rate berdasarkan Gender
sns.barplot(x="sex", y="survived", data=df, ax=axes[1, 0], palette="magma", ci=None)
axes[1, 0].set_title("4. Survival Rate by Sex")
axes[1, 0].set_ylabel("Prosentase Selamat")

# Visualisasi 5: Survival Rate berdasarkan Class
sns.barplot(x="class", y="survived", data=df, ax=axes[1, 1], palette="viridis", ci=None)
axes[1, 1].set_title("5. Survival Rate by Class")
axes[1, 1].set_ylabel("Prosentase Selamat")

# Menghapus subplot kosong (baris 2, kolom 3)
axes[1, 2].axis("off")

plt.tight_layout()
plt.savefig("pertemuan02_eda_lengkap.png")
plt.close()
print("\nVisualisasi disimpan ke: pertemuan02_eda_lengkap.png")

# 5 Insight Berdasarkan Data
print("\n=== 5 Insight Analisis ===")
total_survived = df["survived"].mean()
female_surv = df[df['sex']=='female']['survived'].mean()
male_surv = df[df['sex']=='male']['survived'].mean()
pclass1_surv = df[df['class']=='First']['survived'].mean()

print(f"1. Tingkat survival rata-rata penumpang adalah {total_survived:.1%}.")
print(f"2. Gender Gap: Survival rate perempuan ({female_surv:.1%}) jauh lebih tinggi "
      f"dibanding laki-laki ({male_surv:.1%}), menunjukkan kebijakan 'ladies first'.")
print(f"3. Hierarki Sosial: Penumpang Kelas 1 memiliki peluang selamat tertinggi ({pclass1_surv:.1%}) "
      f"dibandingkan kelas lainnya.")
print(f"4. Preprocessing Skala: Fitur 'age' dan 'fare' kini memiliki rata-rata ~0, "
      f"yang akan mempercepat konvergensi model Machine Learning nantinya.")
print(f"5. Dampak Outlier: Dengan metode clipping pada 'fare', model tidak akan terganggu "
      f"oleh harga tiket ekstrem yang biasanya merupakan anomali.")

print("\n✅ Pengerjaan Pertemuan 02 Selesai.")

/tmp/ipython-input-656/2892159418.py:23: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(x="sex", y="survived", data=df, ax=axes[1, 0], palette="magma", ci=None)
/tmp/ipython-input-656/2892159418.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="sex", y="survived", data=df, ax=axes[1, 0], palette="magma", ci=None)
/tmp/ipython-input-656/2892159418.py:28: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(x="class", y="survived", data=df, ax=axes[1, 1], palette="viridis", ci=None)
/tmp/ipython-input-656/2892159418.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x="cl


Visualisasi disimpan ke: pertemuan02_eda_lengkap.png

=== 5 Insight Analisis ===
1. Tingkat survival rata-rata penumpang adalah 41.4%.
2. Gender Gap: Survival rate perempuan (74.1%) jauh lebih tinggi dibanding laki-laki (21.7%), menunjukkan kebijakan 'ladies first'.
3. Hierarki Sosial: Penumpang Kelas 1 memiliki peluang selamat tertinggi (63.1%) dibandingkan kelas lainnya.
4. Preprocessing Skala: Fitur 'age' dan 'fare' kini memiliki rata-rata ~0, yang akan mempercepat konvergensi model Machine Learning nantinya.
5. Dampak Outlier: Dengan metode clipping pada 'fare', model tidak akan terganggu oleh harga tiket ekstrem yang biasanya merupakan anomali.

✅ Pengerjaan Pertemuan 02 Selesai.
